In [ ]:
import os
from dotenv import load_dotenv

# Force-refresh env vars from .env in case kernel has stale values
load_dotenv('.env', override=True)

from openai import OpenAI
openai_client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))


OPENAI_API_KEY prefix: sk-proj


In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [3]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [4]:
print (len(documents))

72


In [5]:
import minsearch

index = minsearch.Index(
text_fields=["content"],
keyword_fields=["filename"],
)

index.fit(documents)

results = index.search(
query="How does the agentic loop keep calling the model until it stops?",
num_results=5,
)

print("First result filename:", results[0]["filename"])

First result filename: 01-agentic-rag/lessons/14-agentic-loop.md


In [19]:
# Problem 3
from rag_helper import RAGBase

class RAG(RAGBase):
    def search(self, query, num_results=5):
        return index.search(query=query, num_results=num_results)


    def build_context(self, search_results):
        lines = []
        for doc in search_results:
            lines.append(f"File: {doc['filename']}")
            lines.append(doc['content'])
            lines.append('')
        return '\n'.join(lines).strip()

    def llm(self, prompt):
        input_messages = [
            {'role': 'developer', 'content': self.instructions},
            {'role': 'user', 'content': prompt},
        ]
        response = self.llm_client.responses.create(
            model=self.model,
            input=input_messages,
        )
        print(f"Input tokens: {response.usage.input_tokens}")
        return response.output_text

In [25]:
rag = RAG(index=index, llm_client=openai_client)
answer = rag.rag("How does the agentic loop keep calling the model until it stops?")
print(answer)

Input tokens: 7126
It keeps calling the model inside a `while True` loop and stops only when the model returns **no function calls**.

In the code, this is the key logic:

- send the current `messages` to the model
- inspect `response.output`
- if there is a `function_call`, run the tool, append the tool output, and set `has_function_calls = True`
- after processing the response, if `has_function_calls == False`, `break`

So the loop exits when the response contains only a final assistant message and no more tool requests.


In [ ]:
#Problem 4
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
print(f"Number of chunks: {len(chunks)}")

Number of chunks: 295


In [28]:
# Problem 5
# Build an index over chunks and compare token usage vs Problem 3
chunk_index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"],
)
chunk_index.fit(chunks)


class RAGChunk(RAGBase):
    def search(self, query, num_results=5):
        return self.index.search(query=query, num_results=num_results)

    def build_context(self, search_results):
        lines = []
        for doc in search_results:
            lines.append(f"File: {doc['filename']}")
            lines.append(doc['content'])
            lines.append("")
        return "\n".join(lines).strip()

    def llm(self, prompt):
        input_messages = [
            {"role": "developer", "content": self.instructions},
            {"role": "user", "content": prompt},
        ]
        response = self.llm_client.responses.create(
            model="gpt-5.4-mini",
            input=input_messages,
        )
        self.last_input_tokens = response.usage.input_tokens
        print(f"Chunked input tokens: {self.last_input_tokens}")
        return response.output_text


rag_chunk = RAGChunk(index=chunk_index, llm_client=openai_client)
query = "How does the agentic loop keep calling the model until it stops?"
chunk_answer = rag_chunk.rag(query)

problem3_input_tokens = 7126
token_ratio = problem3_input_tokens / rag_chunk.last_input_tokens

print(chunk_answer)
print(f"Problem 3 input tokens: {problem3_input_tokens}")
print(f"Chunked version input tokens: {rag_chunk.last_input_tokens}")
print(f"Token ratio (Problem 3 / Chunked): {token_ratio}")

Chunked input tokens: 2309
The loop keeps calling the model with a `while True` loop, then checks whether the model returned any `function_call` items.

- If there is at least one function call, the code runs the tool, appends the result to `messages`, and loops again.
- If there are no function calls in that turn, `has_function_calls` stays `False`, and the loop breaks.

So the stop condition is: **no function calls this turn means the model is done**.
Problem 3 input tokens: 7126
Chunked version input tokens: 2309
Token ratio (Problem 3 / Chunked): 3.086184495452577


In [29]:
# Problem 6
from toyaikit.tools import Tools
from toyaikit.llm import OpenAIClient
from toyaikit.chat.runners import OpenAIResponsesRunner

search_call_count = 0


def search(query: str, num_results: int = 5) -> list[dict]:
    """Search lesson chunks by query and return the top matching chunk documents."""
    global search_call_count
    search_call_count += 1
    return chunk_index.search(query=query, num_results=num_results)


tools = Tools()
tools.add_tool(search)

runner = OpenAIResponsesRunner(
    tools=tools,
    developer_prompt=(
        "You're a course teaching assistant. "
        "Answer the student's question using the search tool. "
        "Make multiple searches with different keywords before answering."
    ),
    chat_interface=None,
    llm_client=OpenAIClient(model="gpt-5.4-mini", client=openai_client),
)

question = "How does the agentic loop work, and how is it different from plain RAG?"
loop_result = runner.loop(prompt=question, previous_messages=[], callback=None)

print(loop_result.last_message)
print(f"Search tool calls: {search_call_count}")

The **agentic loop** is the pattern where the model can **take multiple turns with tools** until it finishes the task.

In the lesson, it’s described as a loop that:
1. sends the user prompt to the LLM,
2. checks whether the LLM asked to use a tool,
3. runs the tool if needed,
4. sends the tool result back to the LLM,
5. repeats until the model returns a final answer.

So the LLM is effectively “in the driver’s seat,” deciding whether to search again, use another tool, or stop.

### How that differs from plain RAG

**Plain RAG** is usually a **single retrieval + generation pipeline**:

- take the user question,
- search for relevant documents once,
- put that retrieved context into the prompt,
- ask the LLM to answer.

In code from the lesson, it looks like:

```python
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)
```

So plain RAG is basically:
- **one search**
- **one prompt**
- **one answ